In [254]:
import pyomo.environ as pyo
from queue import PriorityQueue
import time
import math
import matplotlib.pyplot as plt
import networkx as nx
from networkx.drawing.nx_agraph import graphviz_layout

# --- 1. Problem Data ---
# Based on the table in the PDF
profits = {
    1: 22, 2: 18, 3: 35, 4: 14, 5: 60, 6: 12,
    7: 50, 8: 15, 9: 21, 10: 19, 11: 24, 12: 16
}
weights = {
    1: 5, 2: 6, 3: 8, 4: 4, 5: 11, 6: 3,
    7: 10, 8: 4, 9: 6, 10: 5, 11: 7, 12: 5
}
# Capacities
capacities = {1: 22, 2: 19}

# Item and Knapsack sets (using n=12)
ITEMS = list(profits.keys())
KNAPSACKS = list(capacities.keys())

In [255]:
def create_base_model():
    """Creates the base Pyomo model structure for the MKP."""
    model = pyo.ConcreteModel()
    
    # Sets
    model.I = pyo.Set(initialize=ITEMS)
    model.J = pyo.Set(initialize=KNAPSACKS)
    
    # Parameters
    model.p = pyo.Param(model.I, initialize=profits)
    model.w = pyo.Param(model.I, initialize=weights)
    model.C = pyo.Param(model.J, initialize=capacities)
    
    # Variables: x_ij
    # We define them as Reals between 0 and 1 for the LP relaxation
    model.x = pyo.Var(model.I, model.J, within=pyo.NonNegativeReals, bounds=(0, 1))
    
    # Objective: Maximize total profit
    def obj_rule(m):
        return sum(m.p[i] * m.x[i, j] for i in m.I for j in m.J)
    model.obj = pyo.Objective(rule=obj_rule, sense=pyo.maximize)
    
    # Constraint 1: Knapsack capacity
    def capacity_rule(m, j):
        return sum(m.w[i] * m.x[i, j] for i in m.I) <= m.C[j]
    model.capacity_con = pyo.Constraint(model.J, rule=capacity_rule)
    
    # Constraint 2: Each item in at most one knapsack
    def item_once_rule(m, i):
        return sum(m.x[i, j] for j in m.J) <= 1
    model.item_once_con = pyo.Constraint(model.I, rule=item_once_rule)
    
    return model

def solve_node_lp(model, solver, bounds_to_apply):
    # Apply the bounds for this specific node
    for (i, j), val in bounds_to_apply.items():
        model.x[i, j].fix(val)
        
    # Solve the LP with load_solutions=False to avoid warnings for infeasible cases
    try:
        results = solver.solve(model, tee=False, load_solutions=False)
    except Exception as e:
        print(f"Solver error: {e}")
        return 'error', None, None, False

    # --- Process Results ---
    
    # Check status
    status = results.solver.termination_condition
    if status == pyo.TerminationCondition.infeasible:
        return 'infeasible', None, None, False
    if status != pyo.TerminationCondition.optimal:
        # print(f"Warning: Node solve was not optimal. Status: {status}")
        return 'suboptimal', None, None, False

    # Load the solution only if optimal
    model.solutions.load_from(results)
    
    # Extract solution
    lp_value = model.obj()
    lp_solution = {}
    is_integer = True
    epsilon = 1e-6  # Tolerance for integrality

    for i in ITEMS:
        for j in KNAPSACKS:
            val = model.x[i, j].value
            if val is None:
                val = 0  # Handle non-assignment
            
            lp_solution[(i, j)] = val
            
            # Check for fractional values
            if abs(val - round(val)) > epsilon:
                is_integer = False
                
    return 'optimal', lp_value, lp_solution, is_integer

## Problem 2

In [256]:
def add_cut(model, min_cover_solution):
    """
    Adds a cover cut to the model.
    Expected input: min_cover_solution is a dictionary {key: 1/0} 
    derived from the minimal cover finder.
    """
    
    # 3. Ensure the model has a container for cuts
    # (It's better to add this in create_base_model, but this is a safety catch)
    if not hasattr(model, 'cuts'):
        model.cuts = pyo.ConstraintList()

    cut_expr = sum(model.x[k] for k in min_cover_solution) <= len(min_cover_solution) - 1
    
    # 5. Add to the list
    model.cuts.add(cut_expr)
    print(f"Added cut covering {len(min_cover_solution)} variables.")

In [257]:
solver = pyo.SolverFactory('gurobi_direct')
model = create_base_model()

In [258]:
is_opt, val, solution, _ = solve_node_lp(model, solver, bounds_to_apply={})
print(val, is_opt)
solution

194.20000000000002 optimal


{(1, 1): 0.0,
 (1, 2): 1.0,
 (2, 1): 0.0,
 (2, 2): 0.0,
 (3, 1): 1.0,
 (3, 2): 0.0,
 (4, 1): 0.0,
 (4, 2): 0.0,
 (5, 1): 1.0,
 (5, 2): 0.0,
 (6, 1): 0.0,
 (6, 2): 1.0,
 (7, 1): 0.0,
 (7, 2): 1.0,
 (8, 1): 0.0,
 (8, 2): 0.0,
 (9, 1): 0.0,
 (9, 2): 0.0,
 (10, 1): 0.6,
 (10, 2): 0.2,
 (11, 1): 0.0,
 (11, 2): 0.0,
 (12, 1): 0.0,
 (12, 2): 0.0}

In [259]:
min_cover1 = [(3,1),(5,1),(10,1)] 
add_cut(model, min_cover1)

Added cut covering 3 variables.


In [260]:
is_opt, val, solution, _ = solve_node_lp(model, solver, bounds_to_apply={})
print(val, is_opt)
solution

194.2 optimal


{(1, 1): 0.6,
 (1, 2): 0.3999999999999999,
 (2, 1): 0.0,
 (2, 2): 0.0,
 (3, 1): 1.0,
 (3, 2): 0.0,
 (4, 1): 0.0,
 (4, 2): 0.0,
 (5, 1): 1.0,
 (5, 2): 0.0,
 (6, 1): 0.0,
 (6, 2): 1.0,
 (7, 1): 0.0,
 (7, 2): 1.0,
 (8, 1): 0.0,
 (8, 2): 0.0,
 (9, 1): 0.0,
 (9, 2): 0.0,
 (10, 1): 0.0,
 (10, 2): 0.8,
 (11, 1): 0.0,
 (11, 2): 0.0,
 (12, 1): 0.0,
 (12, 2): 0.0}

In [261]:
min_cover2 = [(3,1),(5,1),(1,1)] 
add_cut(model, min_cover2)

Added cut covering 3 variables.


In [262]:
is_opt, val, solution, _ = solve_node_lp(model, solver, bounds_to_apply={})
print(val, is_opt)
solution

194.2 optimal


{(1, 1): 1.0,
 (1, 2): 0.0,
 (2, 1): 0.0,
 (2, 2): 0.0,
 (3, 1): 0.0,
 (3, 2): 1.0,
 (4, 1): 0.0,
 (4, 2): 0.0,
 (5, 1): 1.0,
 (5, 2): 0.0,
 (6, 1): 0.0,
 (6, 2): 1.0,
 (7, 1): 0.6,
 (7, 2): 0.3999999999999999,
 (8, 1): 0.0,
 (8, 2): 0.0,
 (9, 1): 0.0,
 (9, 2): 0.0,
 (10, 1): 0.0,
 (10, 2): 0.8000000000000002,
 (11, 1): 0.0,
 (11, 2): 0.0,
 (12, 1): 0.0,
 (12, 2): 0.0}

## Problem 2.2


In [322]:
def sequential_lifting_cut(model, min_cover, solver=None):
    """
    Sequentially lift the base cover inequality for a single knapsack.
    Returns a dictionary alpha[(item, knapsack)] with the lifted coefficients.
    """
    if solver is None:
        solver = pyo.SolverFactory('gurobi_direct')

    knapsack = min_cover[0][1]
    cover = [i for (i, k) in min_cover if k == knapsack]
    rhs = len(cover) - 1

    alpha = {(i, knapsack): 1 if i in cover else 0 for i in ITEMS}

    order = (set(ITEMS) - set(cover))
    # Sort items by solution value (zeros first) then by weight
    order = sorted(list(order), key=lambda i: (model.x[i, knapsack].value if model.x[i, knapsack].value is not None else 0, weights[i]))
    
    for item in order: # Should be sorted more efficently
        residual_capacity = capacities[knapsack] - weights[item]
        if residual_capacity < 0:
            alpha[(item, knapsack)] = rhs
            continue

        sub_model = pyo.ConcreteModel()
        sub_model.COVER = pyo.Set(initialize=cover)
        sub_model.x = pyo.Var(sub_model.COVER, domain=pyo.Binary)

        def obj_rule(m):
            return sum(m.x[k] for k in m.COVER)
        sub_model.obj = pyo.Objective(rule=obj_rule, sense=pyo.maximize)

        def capacity_rule(m):
            return sum(weights[k] * m.x[k] for k in m.COVER) <= residual_capacity
        sub_model.capacity = pyo.Constraint(rule=capacity_rule)

        results = solver.solve(sub_model, tee=False, load_solutions=False)
        if results.solver.termination_condition != pyo.TerminationCondition.optimal:
            raise RuntimeError("Sequential lifting subproblem failed to solve optimally.")

        sub_model.solutions.load_from(results)
        z_val = pyo.value(sub_model.obj)
        alpha[(item, knapsack)] = max(0, rhs - z_val)


    # Lets cut
    if not hasattr(model, 'cuts'):
        model.cuts = pyo.ConstraintList()
        
    cut_expr = sum(alpha[(i, knapsack)] * model.x[i, knapsack] for i in ITEMS) <= rhs
    model.cuts.add(cut_expr)



In [323]:
solver = pyo.SolverFactory('gurobi_direct')
model = create_base_model()

In [324]:
is_opt, val, solution, _ = solve_node_lp(model, solver, bounds_to_apply={})
print(val, is_opt)
solution

194.20000000000002 optimal


{(1, 1): 0.0,
 (1, 2): 1.0,
 (2, 1): 0.0,
 (2, 2): 0.0,
 (3, 1): 1.0,
 (3, 2): 0.0,
 (4, 1): 0.0,
 (4, 2): 0.0,
 (5, 1): 1.0,
 (5, 2): 0.0,
 (6, 1): 0.0,
 (6, 2): 1.0,
 (7, 1): 0.0,
 (7, 2): 1.0,
 (8, 1): 0.0,
 (8, 2): 0.0,
 (9, 1): 0.0,
 (9, 2): 0.0,
 (10, 1): 0.6,
 (10, 2): 0.2,
 (11, 1): 0.0,
 (11, 2): 0.0,
 (12, 1): 0.0,
 (12, 2): 0.0}

In [325]:
min_cover1 = [(3,1),(5,1),(10,1)] 
sequential_lifting_cut(model, min_cover1)

In [326]:
is_opt, val, solution, _ = solve_node_lp(model, solver, bounds_to_apply={})
print(val, is_opt)
solution

194.2 optimal


{(1, 1): 0.2,
 (1, 2): 0.8,
 (2, 1): 0.0,
 (2, 2): 0.0,
 (3, 1): 0.0,
 (3, 2): 1.0,
 (4, 1): 0.0,
 (4, 2): 0.0,
 (5, 1): 1.0,
 (5, 2): 0.0,
 (6, 1): 0.0,
 (6, 2): 1.0,
 (7, 1): 1.0,
 (7, 2): 0.0,
 (8, 1): 0.0,
 (8, 2): 0.0,
 (9, 1): 0.0,
 (9, 2): 0.0,
 (10, 1): 0.0,
 (10, 2): 0.8,
 (11, 1): 0.0,
 (11, 2): 0.0,
 (12, 1): 0.0,
 (12, 2): 0.0}

In [329]:
#min_cover2 = [(1,1),(5,1),(1,1)] 
min_cover2 = [(1,2),(3,2),(6,2),(10,2)]
sequential_lifting_cut(model, min_cover2)

In [330]:
is_opt, val, solution, _ = solve_node_lp(model, solver, bounds_to_apply={})
print(val, is_opt)
solution

194.2 optimal


{(1, 1): 0.0,
 (1, 2): 1.0,
 (2, 1): 0.0,
 (2, 2): 0.0,
 (3, 1): 0.0,
 (3, 2): 1.0,
 (4, 1): 0.0,
 (4, 2): 0.0,
 (5, 1): 0.825,
 (5, 2): 0.17500000000000002,
 (6, 1): 0.975,
 (6, 2): 0.024999999999999977,
 (7, 1): 1.0,
 (7, 2): 0.0,
 (8, 1): 0.0,
 (8, 2): 0.0,
 (9, 1): 0.0,
 (9, 2): 0.0,
 (10, 1): 0.0,
 (10, 2): 0.7999999999999999,
 (11, 1): 0.0,
 (11, 2): 0.0,
 (12, 1): 0.0,
 (12, 2): 0.0}